# go-evo: PPO self-play training, step by step

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrimozRavbar/go-evo/blob/main/notebooks/ppo_training_colab.ipynb)

The whole PPO training of phase 5 is written out in this notebook: network, value head, self-play rollouts, credit assignment, the PPO update, the training loop, checkpoints and evaluation.
Only the Go engine (`goevo.engine` / `goevo.fastgo`, numba) and the input encoding (`goevo.player.encode_planes`) are imported. `goevo/ppo.py` is the same algorithm as a script.

**Setup**
* 19×19, komi 7.5, area scoring.
* Pass only when no stone move is possible; never fill your own single-point eye.
* Reward ±1 at the end of the game.
* Self-play against the current policy and a pool of past snapshots.
* CPU only. Rollouts use worker processes, and the update uses all cores.

Defaults: the phase 5 network (16 channels, 4 blocks, ~20.5k policy weights) and a 1-hour budget. Set `CHANNELS=32, BLOCKS=6, SYM=True` for the phase 5b setup.
**Save to Google Drive** (cell 1) so that training survives a Colab disconnect. Re-running the training cell resumes from the last checkpoint.

## 0. Setup

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"
if IN_COLAB:
    if not os.path.exists("go-evo"):
        !git clone -q --depth 1 -b {BRANCH} https://github.com/PrimozRavbar/go-evo.git
    %cd go-evo
    !pip install -q numba
else:
    os.chdir(os.path.abspath(".."))  # repo root
    sys.path.insert(0, os.getcwd())

import json, time, multiprocessing as mp
from dataclasses import dataclass, asdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

from goevo.engine import make_game, BLACK, WHITE
from goevo.match import GameConfig, own_eye_mask
from goevo.player import encode_planes, CONV_PLANES, NetConfig, Player, RandomPlayer, load_population, save_population
print("CPU cores:", os.cpu_count(), "| torch", torch.__version__)

## 1. Settings

In [ ]:
# --- network
CHANNELS = 16  # @param {type:"integer"}
BLOCKS = 4  # @param {type:"integer"}
# --- self-play
GAMES_PER_ITER = 32  # @param {type:"integer"}
P_SELF = 0.5  # @param {type:"number"}
SNAPSHOT_EVERY = 10  # @param {type:"integer"}
POOL_SIZE = 50  # @param {type:"integer"}
SYM = False  # @param {type:"boolean"}
# --- PPO
LR = 3e-4  # @param {type:"number"}
EPOCHS = 3  # @param {type:"integer"}
MINIBATCH = 1024  # @param {type:"integer"}
CLIP = 0.2  # @param {type:"number"}
VF_COEF = 0.5  # @param {type:"number"}
ENT_COEF = 0.01  # @param {type:"number"}
MAX_GRAD = 0.5  # @param {type:"number"}
TARGET_KL = 0.03  # @param {type:"number"}
LAM = 0.95  # @param {type:"number"}
# --- run
HOURS = 1.0  # @param {type:"number"}
WORKERS = 0  # @param {type:"integer"}
SEED = 0  # @param {type:"integer"}
RUN_NAME = "ppo_nb_s0"  # @param {type:"string"}
USE_DRIVE = False  # @param {type:"boolean"}

WORKERS = WORKERS or os.cpu_count()
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"/content/drive/MyDrive/go-evo/{RUN_NAME}"
else:
    OUT = f"work/{RUN_NAME}"
os.makedirs(OUT, exist_ok=True)
N = 19
GC = GameConfig(board_size=N).resolved()  # komi 7.5, move cap 3*N*N, forced pass, no own-eye fill
print("output:", OUT, "| workers:", WORKERS)

## 2. The network
The residual CNN of phase 4:
* a 3×3 stem conv;
* `BLOCKS` pre-activation blocks (GroupNorm(1) → tanh → conv → GroupNorm(1) → tanh → conv, plus the skip connection);
* a final norm and tanh;
* **policy:** a 1×1 conv gives one logit per point, and a linear layer on the board average gives the pass logit;
* **value:** board average → 16 tanh → 1 tanh, an estimate of the result for the player to move (−1 … +1).

The policy parameters are in the same order as the evolved genomes. So `policy_genome()` gives a flat vector that `goevo.player.Player` can play, against the evolved players or in the games notebooks.

In [ ]:
GAIN = 5 / 3  # tanh

class ResBlock(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.n1, self.c1 = nn.GroupNorm(1, C), nn.Conv2d(C, C, 3, padding=1)
        self.n2, self.c2 = nn.GroupNorm(1, C), nn.Conv2d(C, C, 3, padding=1)
    def forward(self, x):
        return x + self.c2(torch.tanh(self.n2(self.c1(torch.tanh(self.n1(x))))))

class Policy(nn.Module):
    """Same layout (and parameter order) as goevo.player.ResNet."""
    def __init__(self, C, blocks):
        super().__init__()
        self.stem = nn.Conv2d(CONV_PLANES, C, 3, padding=1)
        self.blocks = nn.ModuleList([ResBlock(C) for _ in range(blocks)])
        self.norm = nn.GroupNorm(1, C)
        self.point = nn.Conv2d(C, 1, 1)
        self.pass_ = nn.Linear(C, 1)
    def features(self, x):
        h = self.stem(x)
        for b in self.blocks:
            h = b(h)
        return torch.tanh(self.norm(h))
    def forward(self, x):
        f = self.features(x)
        return torch.cat([self.point(f).flatten(1), self.pass_(f.mean((2, 3)))], 1)

class PolicyValue(nn.Module):
    def __init__(self, C=CHANNELS, blocks=BLOCKS, vh=16):
        super().__init__()
        self.C, self.nblocks = C, blocks
        self.pi = Policy(C, blocks)
        self.v1, self.v2 = nn.Linear(C, vh), nn.Linear(vh, 1)
    def forward(self, x):
        f = self.pi.features(x)
        g = f.mean((2, 3))
        logits = torch.cat([self.pi.point(f).flatten(1), self.pi.pass_(g)], 1)
        value = torch.tanh(self.v2(torch.tanh(self.v1(g)))).squeeze(1)
        return logits, value
    def policy_genome(self):
        return nn.utils.parameters_to_vector(self.pi.parameters()).detach().numpy().astype(np.float32)

@torch.no_grad()
def init_like_evolution(m: PolicyValue, seed=0):
    """Same random init as an evolution genome: variance-preserving convs, second conv of each
    block at 0.1x, norm gain 1 / bias 0, biases N(0, 0.1). Small value head."""
    g = torch.Generator().manual_seed(seed)
    def conv(c, k=1.0):
        fan_in = c.weight[0].numel()
        c.weight.normal_(0, k * GAIN / fan_in ** 0.5, generator=g)
        c.bias.normal_(0, 0.1 * k, generator=g)
    for mod in m.pi.modules():
        if isinstance(mod, nn.GroupNorm):
            mod.weight.fill_(1.0); mod.bias.zero_()
    conv(m.pi.stem)
    for b in m.pi.blocks:
        conv(b.c1); conv(b.c2, 0.1)
    for lin in (m.pi.point, m.pi.pass_):
        lin.weight.normal_(0, GAIN / m.C ** 0.5, generator=g); lin.bias.normal_(0, 0.1, generator=g)
    m.v1.weight.normal_(0, 1 / m.C ** 0.5, generator=g); m.v1.bias.zero_()
    m.v2.weight.normal_(0, 0.1 / 16 ** 0.5, generator=g); m.v2.bias.zero_()
    return m

net = init_like_evolution(PolicyValue(), SEED)
n_pi = sum(p.numel() for p in net.pi.parameters())
print(f"policy {n_pi:,} params, value head {sum(p.numel() for p in [*net.v1.parameters(), *net.v2.parameters()]):,}")
# check: the flat policy vector plays identically in goevo's Player
NET_CFG = NetConfig(board_size=N, arch="resnet", channels=CHANNELS, blocks=BLOCKS)
x = torch.from_numpy(encode_planes(make_game(N), make_game(N).legal_mask()))[None]
with torch.no_grad():
    print("matches goevo Player:", np.allclose(net(x)[0][0].numpy(), Player(net.policy_genome(), NET_CFG).logits(x[0].numpy()), atol=1e-4))

## 3. Rules, inputs and board symmetries
* **Inputs:** 11 planes from the mover's view (`encode_planes`): own / opponent / empty points, legal moves, own and opponent groups with 1 / 2 / ≥3 liberties, and a plane of ones.
* **Allowed moves:** legal, minus filling your own single-point eye. Pass only when nothing else is allowed.
* **Symmetries:** with `SYM`, every move is decided on a randomly rotated or mirrored board and stored in that orientation. This is exact PPO data, and all 8 orientations are seen.

In [ ]:
def allowed_moves(game, legal):
    allowed = legal.copy()
    eyes = game.own_eye_mask() if hasattr(game, "own_eye_mask") else own_eye_mask(game)
    allowed[:-1] &= ~eyes
    allowed[-1] = not allowed[:-1].any()   # pass only when forced
    if not allowed.any():
        allowed[-1] = True
    return allowed

def sym(x, k):  # k-th of the 8 symmetries on the last two axes
    if k >= 4:
        x = x[..., ::-1]
    return np.rot90(x, k % 4, axes=(-2, -1))

PERMS = [np.ascontiguousarray(sym(np.arange(N * N).reshape(N, N), k)).ravel() for k in range(8)]
# PERMS[k][j] = the original point shown at point j of the transformed board
for k in range(8):
    t = np.random.rand(3, N, N)
    assert np.allclose(np.ascontiguousarray(sym(t, k)).reshape(3, -1), t.reshape(3, -1)[:, PERMS[k]])
print("symmetries ok")

## 4. Self-play rollouts
One game per call, run in worker processes. Moves are **sampled** from the softmax over the allowed moves (temperature 1).

The opponent is either the current policy, in which case both sides are training data, or a past snapshot, in which case only the learner's side is used.

For each learner move we store the input planes, the allowed mask, the action, its log-probability and the value estimate. The reward is the game result from that side's view.

In [ ]:
def make_net(C, blocks, sd):
    m = PolicyValue(C, blocks)
    m.load_state_dict(sd)
    return m.eval()

def sample(logits, allowed, rng):
    z = np.where(allowed, logits.astype(np.float64), -np.inf)
    p = np.exp(z - z.max()); p /= p.sum()
    a = int(rng.choice(len(p), p=p))
    return a, float(np.log(p[a]))

def rollout(job):
    sd, opp_sd, learner_color, seed, C, blocks, use_sym = job
    torch.set_num_threads(1)
    rng = np.random.default_rng(seed)
    me = make_net(C, blocks, sd)
    other = None if opp_sd is None else make_net(C, blocks, opp_sd)
    game = make_game(N, GC.komi, GC.backend)
    traj = {BLACK: [], WHITE: []}
    with torch.no_grad():
        while not game.is_over() and game.move_number < GC.max_moves:
            legal = game.legal_mask()
            allowed = allowed_moves(game, legal)
            x = encode_planes(game, legal)
            if use_sym:
                k = int(rng.integers(8))
                x = np.ascontiguousarray(sym(x, k))
                allowed = np.append(allowed[:-1][PERMS[k]], allowed[-1])
            color = game.to_play
            learner = other is None or color == learner_color
            logits, v = (me if learner else other)(torch.from_numpy(x)[None])
            a, lp = sample(logits[0].numpy(), allowed, rng)
            if learner:
                traj[color].append((x.astype(np.int8), allowed, a, lp, float(v[0])))
            game.play(int(PERMS[k][a]) if use_sym and a < N * N else a)
    winner = game.winner()
    out = []
    for color, steps in traj.items():
        if steps:
            X, M, A, LP, V = map(np.array, zip(*steps))
            out.append(dict(X=X, M=M, A=A.astype(np.int64), LP=LP.astype(np.float32),
                            V=V.astype(np.float32), R=float(winner * color), vs_self=other is None))
    return out, game.move_number

sd = {k: v.clone() for k, v in net.state_dict().items()}
t0 = time.time()
trajs, moves = rollout((sd, None, BLACK, 0, CHANNELS, BLOCKS, SYM))
print(f"one self-play game: {moves} moves, {time.time() - t0:.1f}s, "
      f"{[len(t['A']) for t in trajs]} training steps, rewards {[t['R'] for t in trajs]}")

## 5. Credit assignment (GAE)
With a single ±1 at the end of a roughly 450-move game, the value head tells PPO which moves made the position better or worse than expected:

δₜ = Vₜ₊₁ − Vₜ (plus the reward on the last move), Aₜ = Σ (λ)ᵏ δₜ₊ₖ, with γ = 1.

The value target is Aₜ + Vₜ.

In [ ]:
def gae(values, reward, lam=LAM):
    delta = np.append(values[1:], 0.0) - values
    delta[-1] += reward
    adv = np.zeros_like(values)
    acc = 0.0
    for t in range(len(values) - 1, -1, -1):
        acc = delta[t] + lam * acc
        adv[t] = acc
    return adv, adv + values

a, r = gae(np.array([0.0, 0.1, 0.3, 0.6], np.float32), +1.0)
print("advantages", a.round(3), " returns", r.round(3))

## 6. The PPO update
For each minibatch:
* **Policy:** the clipped objective −min(ρA, clip(ρ, 1±ε)A), where ρ = π_new(a)/π_old(a).
* **Value:** a squared-error loss against the returns.
* **Exploration:** an entropy bonus over the allowed moves.
* **Stability:** gradients are clipped, and if the mean KL to the old policy exceeds `TARGET_KL`, the remaining epochs are skipped.

Advantages are normalized per batch.

In [ ]:
def ppo_update(model, opt, X, M, A, LP, ADV, RET):
    ADV = (ADV - ADV.mean()) / (ADV.std() + 1e-8)
    stats, n = [], len(A)
    for epoch in range(EPOCHS):
        perm, kls = np.random.permutation(n), []
        for s in range(0, n, MINIBATCH):
            i = perm[s:s + MINIBATCH]
            x = torch.from_numpy(X[i].astype(np.float32))
            m, act = torch.from_numpy(M[i]), torch.from_numpy(A[i])
            old, adv, ret = torch.from_numpy(LP[i]), torch.from_numpy(ADV[i]), torch.from_numpy(RET[i])
            logits, v = model(x)
            logp_all = F.log_softmax(logits.masked_fill(~m, -1e9), 1)
            logp = logp_all.gather(1, act[:, None])[:, 0]
            ratio = torch.exp(logp - old)
            pg = -torch.min(ratio * adv, ratio.clamp(1 - CLIP, 1 + CLIP) * adv).mean()
            vl = F.mse_loss(v, ret)
            ent = -(logp_all.exp() * logp_all.masked_fill(~m, 0.0)).sum(1).mean()
            loss = pg + VF_COEF * vl - ENT_COEF * ent
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD)
            opt.step()
            kls.append((old - logp).mean().item())
            stats.append((pg.item(), vl.item(), ent.item(), kls[-1]))
        if np.mean(kls) > TARGET_KL:
            break
    s = np.mean(stats, 0)
    return dict(pg=s[0], vl=s[1], ent=s[2], kl=s[3], epochs=epoch + 1)

## 7. Evaluation helpers
Deterministic play (the top allowed move) with `goevo.player.Player`, the same way the evolved players were evaluated.

In [ ]:
from goevo.parallel import _play   # plays one game between two player specs

def match(pool, genome, opponents, games_each=2, seed=0):
    """Win rate of `genome` vs a list of (genome, NetConfig) or 'random', both colours."""
    me = ("genome", "me", genome, NET_CFG)
    jobs, signs = [], []
    for j, opp in enumerate(opponents):
        o = ("random", "rm") if opp == "random" else ("genome", f"o{j}", opp[0], opp[1])
        for k in range(games_each):
            jobs.append((me, o, NET_CFG, GC, seed * 1000 + len(jobs), False) if k % 2 == 0 else
                        (o, me, NET_CFG, GC, seed * 1000 + len(jobs), False))
            signs.append(1 if k % 2 == 0 else -1)
    recs = pool.map(_play, jobs, chunksize=1)
    return float(np.mean([r.winner == s for r, s in zip(recs, signs)]))

def champions(path):
    if not os.path.exists(path):
        return []
    gs, cfg, *_ = load_population(path)
    return [(g, cfg) for g in gs]

REFS = {"phase 4 evolution": champions("runs/p4_s0/gen4000.pt"),
        "phase 5 PPO": champions("runs/p5_s0/final.pt")}
print({k: len(v) for k, v in REFS.items()})

## 8. Training loop
Each iteration:
1. Play `GAMES_PER_ITER` self-play games in parallel.
2. Compute GAE.
3. Run the PPO update.
4. Every `SNAPSHOT_EVERY` iterations, add a snapshot to the opponent pool.

Every 25 iterations it plays the random mover and the phase 4 / phase 5 champions (deterministic), and the plot is redrawn.

Everything is saved to `OUT/state.pt` every 5 iterations. **Re-run this cell to continue after a stop or a disconnect.** The time budget counts training time only.

In [ ]:
model = init_like_evolution(PolicyValue(), SEED)
opt = torch.optim.Adam(model.parameters(), lr=LR)
rng = np.random.default_rng(SEED)
S = dict(it=0, elapsed=0.0, games=0, hist=[], pool=[{k: v.clone() for k, v in model.state_dict().items()}])
if os.path.exists(f"{OUT}/state.pt"):
    st = torch.load(f"{OUT}/state.pt", weights_only=False)
    model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"]); rng.bit_generator.state = st["rng"]
    S = st["S"]
    print(f"resumed at iteration {S['it']} ({S['elapsed'] / 3600:.2f} h used)")

def save_state():
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "rng": rng.bit_generator.state, "S": S},
               f"{OUT}/state.pt.tmp")
    os.replace(f"{OUT}/state.pt.tmp", f"{OUT}/state.pt")

def plot(h):
    it = [r["it"] for r in h]
    fig, ax = plt.subplots(1, 4, figsize=(17, 3.2))
    pw = [r["pool_wr"] for r in h]
    w = min(20, len(pw))
    ax[0].plot(it, pw, alpha=.3); ax[0].plot(it, np.convolve(pw, np.ones(w) / w, "same")); ax[0].set_title("win rate vs pool snapshots")
    ax[1].plot(it, [r["ent"] for r in h]); ax[1].set_title("policy entropy")
    ax[2].plot(it, [r["vl"] for r in h]); ax[2].set_title("value loss")
    ev = [r for r in h if "eval" in r]
    for k in (ev[0]["eval"] if ev else {}):
        ax[3].plot([r["it"] for r in ev], [r["eval"][k] for r in ev], "o-", label=k)
    ax[3].set_ylim(-0.05, 1.05); ax[3].set_title("deterministic win rate"); ax[3].legend(fontsize=7)
    for a in ax: a.grid(alpha=.3); a.set_xlabel("iteration")
    plt.tight_layout(); plt.show()

pool = mp.get_context("fork").Pool(WORKERS)
torch.set_num_threads(WORKERS)
try:
    while S["elapsed"] < HOURS * 3600:
        t0 = time.time()
        sd = {k: v.clone() for k, v in model.state_dict().items()}
        jobs = []
        for k in range(GAMES_PER_ITER):
            opp = None if rng.random() < P_SELF else S["pool"][int(rng.integers(len(S["pool"])))]
            jobs.append((sd, opp, BLACK if k % 2 == 0 else WHITE, int(rng.integers(2 ** 31)), CHANNELS, BLOCKS, SYM))
        res = pool.map(rollout, jobs, chunksize=1)
        trajs = [t for r in res for t in r[0]]
        t_play = time.time() - t0
        X = np.concatenate([t["X"] for t in trajs]); M = np.concatenate([t["M"] for t in trajs])
        A = np.concatenate([t["A"] for t in trajs]); LP = np.concatenate([t["LP"] for t in trajs])
        adv, ret = zip(*[gae(t["V"], t["R"]) for t in trajs])
        st = ppo_update(model, opt, X, M, A, LP, np.concatenate(adv), np.concatenate(ret))
        S["it"] += 1; S["games"] += GAMES_PER_ITER
        if S["it"] % SNAPSHOT_EVERY == 0:
            S["pool"] = (S["pool"] + [{k: v.clone() for k, v in model.state_dict().items()}])[-POOL_SIZE:]
        S["elapsed"] += time.time() - t0
        pw = [t["R"] > 0 for t in trajs if not t["vs_self"]]
        rec = dict(it=S["it"], hours=S["elapsed"] / 3600, games=S["games"], steps=len(A),
                   moves=float(np.mean([r[1] for r in res])), pool_wr=float(np.mean(pw)) if pw else 0.5,
                   t_play=t_play, t_train=time.time() - t0 - t_play, **st)
        if S["it"] % 25 == 0 or S["it"] == 1:
            g = model.policy_genome()
            rec["eval"] = {"random mover": match(pool, g, ["random"], 8, S["it"])}
            for name, opps in REFS.items():
                if opps:
                    rec["eval"][name] = match(pool, g, opps, 2, S["it"])
        S["hist"].append(rec)
        if S["it"] % 5 == 0:
            save_state()
        clear_output(wait=True)
        print(f"it {S['it']}  {S['elapsed'] / 3600:.2f}/{HOURS} h  games {S['games']}  steps {len(A)}  "
              f"moves {rec['moves']:.0f}  pool_wr {rec['pool_wr']:.2f}  ent {st['ent']:.2f}  kl {st['kl']:.3f}  "
              f"vl {st['vl']:.3f}  ({t_play:.1f}s play + {rec['t_train']:.1f}s train)")
        if "eval" in [k for r in S["hist"] for k in r]:
            print("last eval:", next(r["eval"] for r in reversed(S["hist"]) if "eval" in r))
        plot(S["hist"])
finally:
    save_state()
    pool.close(); pool.join()
print("done:", S["it"], "iterations,", S["games"], "games,", f"{S['elapsed'] / 3600:.2f} h")

## 9. Save the trained network
* `final.pt`: the last 4 policies as genomes (the same format as the evolved runs). It can be loaded in the games notebooks or by `goevo.eval_phase5`.
* `full.pt`: the whole network, policy plus value head, as a torch state_dict for analysis. It is compatible with `goevo.ppo.load_full`.

In [ ]:
last = [PolicyValue() for _ in range(3)]
for m, sd in zip(last, S["pool"][-3:]):
    m.load_state_dict(sd)
finals = ([m.policy_genome() for m in last] + [model.policy_genome()])[-4:]
save_population(f"{OUT}/final.pt", finals, NET_CFG, [f"final{k}" for k in range(len(finals))],
                {"it": S["it"], "games": S["games"], "hours": S["elapsed"] / 3600})
torch.save({"cfg": asdict(NET_CFG), "state_dict": model.state_dict(),
            "extra": {"it": S["it"], "games": S["games"], "hours": S["elapsed"] / 3600}}, f"{OUT}/full.pt")
json.dump(S["hist"], open(f"{OUT}/history.json", "w"))
print("saved", OUT, os.listdir(OUT))

## 10. Final evaluation and a game to watch

In [ ]:
EVAL_GAMES = 4  # @param {type:"integer"}
with mp.get_context("fork").Pool(WORKERS) as p:
    g = model.policy_genome()
    print(f"vs random mover: {match(p, g, ['random'], 50):.2f}  (50 games)")
    for name, opps in REFS.items():
        if opps:
            print(f"vs {name}: {match(p, g, opps, EVAL_GAMES):.2f}  ({EVAL_GAMES * len(opps)} games)")

In [ ]:
from goevo.match import play_game
from goevo.viz import animate_game
OPPONENT = "phase 4 evolution"  # @param ["phase 4 evolution", "phase 5 PPO", "random mover"]
me = Player(model.policy_genome(), NET_CFG, name="this run")
opp = RandomPlayer("random mover") if OPPONENT == "random mover" else Player(REFS[OPPONENT][0][0], REFS[OPPONENT][0][1], name=OPPONENT)
rec = play_game(me, opp, GameConfig(board_size=N))
print(f"{rec.black} (B) vs {rec.white} (W): {rec.result_string}, {len(rec.moves)} moves")
animate_game(rec, interval_ms=120, every=3)